In [7]:
from pathlib import Path
import pandas as pd 
from langchain_core.documents import Document
from langchain_community.document_loaders import (
    PyPDFLoader,
    BSHTMLLoader
)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings



from langchain_community.vectorstores import FAISS

# from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

# from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec


3. Project paths

In [8]:
BASE_DIR = Path(
    r"C:\Users\USER\OneDrive\Desktop\AI-Financial-Risk-Intelligence"
)

DOCUMENT_DIR = (
    BASE_DIR
    / "data"
    / "raw"
    / "Documents"
)

print(DOCUMENT_DIR)

C:\Users\USER\OneDrive\Desktop\AI-Financial-Risk-Intelligence\data\raw\Documents


4. Load PDF documents using LangChain

In [9]:
def load_pdf_documents(folder):

    documents = []

    for pdf_path in Path(folder).rglob("*.pdf"):

        loader = PyPDFLoader(
            str(pdf_path)
        )

        docs = loader.load()

        for doc in docs:

            doc.metadata["source_file"] = (
                pdf_path.name
            )

            doc.metadata["file_path"] = (
                str(pdf_path)
            )

            doc.metadata["document_type"] = (
                "PDF"
            )

        documents.extend(docs)

    return documents

In [10]:
pdf_documents = load_pdf_documents(
    DOCUMENT_DIR
)

print(
    "PDF document units:",
    len(pdf_documents)
)

PDF document units: 380


5. Load HTML documents

In [11]:
def load_html_documents(folder):

    documents = []

    for html_path in Path(folder).rglob("*.html"):

        loader = BSHTMLLoader(
            str(html_path)
        )

        docs = loader.load()

        for doc in docs:

            doc.metadata["source_file"] = (
                html_path.name
            )

            doc.metadata["file_path"] = (
                str(html_path)
            )

            doc.metadata["document_type"] = (
                "HTML"
            )

        documents.extend(docs)

    return documents

In [12]:
html_documents = load_html_documents(
    DOCUMENT_DIR
)

print(
    "HTML document units:",
    len(html_documents)
)

c:\Users\USER\anaconda3\envs\ds\Lib\site-packages\langchain_community\document_loaders\html_bs.py:126: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(f, **self.bs_kwargs)


HTML document units: 5


6. Combine all documents

In [13]:
documents = (
    pdf_documents
    + html_documents
)

print(
    "Total loaded documents:",
    len(documents)
)

Total loaded documents: 385


In [14]:
for doc in documents[:3]:

    print("=" * 80)

    print(
        "Source:",
        doc.metadata.get("source_file")
    )

    print(
        "Page:",
        doc.metadata.get("page")
    )

    print(
        doc.page_content[:1000]
    )

Source: DOC-01_American_Express_2025_Annual_Report.pdf
Page: 0

Source: DOC-01_American_Express_2025_Annual_Report.pdf
Page: 1

Source: DOC-01_American_Express_2025_Annual_Report.pdf
Page: 2



7. Add organization/category metadata

In [15]:
def identify_source(filename):

    filename = filename.lower()

    if "american" in filename:
        return "American Express"

    if "rbi" in filename:
        return "RBI"

    if "sebi" in filename:
        return "SEBI"

    if "fatf" in filename:
        return "FATF"

    return "Unknown"

In [16]:
for doc in documents:

    source_file = doc.metadata.get(
        "source_file",
        ""
    )

    doc.metadata["organization"] = (
        identify_source(source_file)
    )

8. Text splitting

In [17]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

In [18]:
chunks = text_splitter.split_documents(
    documents
)

print(
    "Total chunks:",
    len(chunks)
)

Total chunks: 2508


In [19]:
for i, chunk in enumerate(chunks[:5]):

    print("=" * 80)

    print("Chunk:", i)

    print(
        "Source:",
        chunk.metadata.get("source_file")
    )

    print(
        "Page:",
        chunk.metadata.get("page")
    )

    print(
        chunk.page_content[:500]
    )

Chunk: 0
Source: DOC-01_American_Express_2025_Annual_Report.pdf
Page: 3
2025 was an excellent year for American Express. Propelled 
by our loyal customers, our global network of merchants and 
partners, and our talented colleagues, we delivered some of 
the best financial results in our long history, building on the 
strong growth we have sustained since introducing our long-
term growth aspirations in 2022.
Our consistently strong performance has been powered by 
our Framework for Winning, a strategic roadmap we have had 
in place since 2018 that lays out our visio
Chunk: 1
Source: DOC-01_American_Express_2025_Annual_Report.pdf
Page: 3
can discover products and services, make decisions, 
and complete transactions on behalf of consumers and 
businesses – from booking travel and making dinner 
reservations to replenishing business inventories, managing 
expenses, and completing payments autonomously.
STEPHEN J. SQUERI, CHAIRMAN & CEO
Our business is driven by the value we create through

9. HuggingFace embedding model

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)